# Sinusoidal Positional Encoding — Solution

Reference implementation for `01_sinusoidal_exercise.ipynb`.

Each solution cell is followed by commentary explaining the design choices.

$$PE_{(pos,\, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos,\, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

**Why does `i` only range over `d_model/2`, not the full `d_model`?**

`i` indexes dimension *pairs*, not individual dimensions. Each value of `i` produces **two** output dimensions:

- Dim `2i` (even index) gets $\sin(\omega_i \cdot pos)$
- Dim `2i+1` (odd index) gets $\cos(\omega_i \cdot pos)$

So one `i` = one frequency $\omega_i$ = one (sin, cos) pair = 2 dimensions. To fill all `d_model` columns of the PE matrix, you need exactly `d_model / 2` values of `i`.

| `i` | dim `2i` | dim `2i+1` |
|-----|----------|------------|
| 0   | 0 (sin)  | 1 (cos)    |
| 1   | 2 (sin)  | 3 (cos)    |
| ... | ...      | ...        |
| `d/2 - 1` | `d-2` (sin) | `d-1` (cos) |

If `i` went to `d_model` you'd be writing to columns that don't exist.

**The deeper reason — it's about rotations.** Each pair $(2i, 2i+1)$ is **one 2D rotation** at frequency $\omega_i$. A 2D rotation is fully described by a sin/cos pair — you can't have a rotation in 1D. So:

$$\text{1 frequency} = \text{1 rotation} = \text{1 (sin, cos) pair} = \text{2 dimensions}$$

This is also why **`d_model` must be even** for sinusoidal PE to work cleanly — odd `d_model` would leave a half-pair with no partner. The same constraint applies to `d_head` in RoPE (Part 2), for the exact same reason.

## Setup

In [1]:
import torch 
import torch.nn as nn

## Solution 1 — Frequencies

In [2]:
def get_frequencies(d_model: int) -> 'torch.Tensor':
    """Return shape (d_model // 2,) tensor of frequencies omega_i."""

    ## torch.arange() -> already gives a vector
    i = torch.arange(d_model//2)
    omega = 1 / torch.pow(10000,2*i/d_model)

    ### for numerical stability
    ## omega = torch.exp(-math.log(10000)*2*i/d_model)
    
    return omega

In [3]:
get_frequencies(4)

tensor([1.0000, 0.0100])

## Solution 2 — Build the PE matrix

In [4]:
def sinusoidal_pe(max_len: int, d_model: int) -> 'torch.Tensor':
    """Return shape (max_len, d_model) sinusoidal positional encoding."""
    ## Output Shape - [max_len,d_model]
    ## Omegs shape - [d_model//2,]
    ## pos_shape - [max_len,]
    pos_embedding = torch.zeros((max_len,d_model))
    omega = get_frequencies(d_model)
    pos_indexes= torch.arange(max_len)

    angles = pos_indexes[:,None] * omega[None,:] ## shape max_leng,1*1,d_model//2 -> max_len*d_model//2 after broadcasting

    pos_embedding[:,::2] = torch.sin(angles)
    pos_embedding[:,1::2] = torch.cos(angles)
    # for i in range(d_model):
    #     pos_embedding[i,::2] = torch.sin(i*omega[i])
    #     pos_embedding[i,1::2] = torch.cos(i*omega[i])

    
    # return pos_embedding

    return pos_embedding

In [5]:
sinusoidal_pe(4,8)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00]])

**Why this design?** _Explain the angle matrix trick, interleaving sin/cos with slicing, and why this is more efficient than a Python loop._

## Solution 3 — Rotation property verification

### Deriving $R_k$ from scratch

We want a matrix $R_k$ such that, for any dimension pair $(2i, 2i+1)$:

$$\begin{bmatrix} PE(pos+k,\, 2i) \\ PE(pos+k,\, 2i+1) \end{bmatrix} = R_k \cdot \begin{bmatrix} PE(pos,\, 2i) \\ PE(pos,\, 2i+1) \end{bmatrix}$$

In our convention, the pair stores $[\sin(\omega_i \cdot pos),\; \cos(\omega_i \cdot pos)]$ — sin first, cos second. Substitute the formula:

$$\begin{bmatrix} \sin(\omega_i(pos+k)) \\ \cos(\omega_i(pos+k)) \end{bmatrix} = R_k \cdot \begin{bmatrix} \sin(\omega_i \cdot pos) \\ \cos(\omega_i \cdot pos) \end{bmatrix}$$

**Step 1 — apply the angle-addition formulas to the LHS.**

$$\sin(\omega_i \cdot pos + \omega_i \cdot k) = \sin(\omega_i \cdot pos)\cos(\omega_i \cdot k) + \cos(\omega_i \cdot pos)\sin(\omega_i \cdot k)$$

$$\cos(\omega_i \cdot pos + \omega_i \cdot k) = \cos(\omega_i \cdot pos)\cos(\omega_i \cdot k) - \sin(\omega_i \cdot pos)\sin(\omega_i \cdot k)$$

**Step 2 — recognize that each line is a linear combination of $\sin(\omega_i \cdot pos)$ and $\cos(\omega_i \cdot pos)$.** Let $\phi = \omega_i \cdot k$ for brevity:

$$\sin(\omega_i(pos+k)) = \cos(\phi) \cdot \sin(\omega_i \cdot pos) + \sin(\phi) \cdot \cos(\omega_i \cdot pos)$$

$$\cos(\omega_i(pos+k)) = -\sin(\phi) \cdot \sin(\omega_i \cdot pos) + \cos(\phi) \cdot \cos(\omega_i \cdot pos)$$

**Step 3 — read off the coefficients.** The first row's coefficients are $[\cos\phi,\; \sin\phi]$. The second row's are $[-\sin\phi,\; \cos\phi]$. So:

$$R_k = \begin{bmatrix} \cos(\omega_i \cdot k) & \sin(\omega_i \cdot k) \\ -\sin(\omega_i \cdot k) & \cos(\omega_i \cdot k) \end{bmatrix}$$

**Sign convention gotcha.** The standard textbook 2D rotation matrix is $\begin{bmatrix}\cos & -\sin \\ \sin & \cos\end{bmatrix}$ — that's correct when your vector stores $[\cos, \sin]$ (cos first). Sinusoidal PE stores $[\sin, \cos]$ (sin first), which flips the off-diagonal signs. Easy to get wrong.

**Two key properties of this $R_k$:**

1. **Depends only on $k$, not on $pos$.** The same matrix transforms position 3 → position 8 and position 100 → position 105. The model can learn one $R_k$ per offset and reuse it everywhere.
2. **Block-diagonal across dimension pairs.** The full $R_k$ for the entire `d_model`-dimensional vector is $d/2$ such 2×2 blocks stacked along the diagonal — each pair rotates independently at its own frequency $\omega_i$.

This rotation property is the seed of RoPE — covered in Part 2.

In [6]:
d_model = 8 ##let
max_len = 8 ##let
omegas = get_frequencies(d_model)

def check(pos,k,embed_dim):
    relative_angle = omegas[embed_dim]*k
    cosine,sine = torch.cos(relative_angle),torch.sin(relative_angle)

    R_k = torch.tensor([[cosine,sine],[-sine,cosine]])

    PE = sinusoidal_pe(max_len,d_model)

    print(PE[pos+k,2*embed_dim:2*embed_dim+2])
    print(R_k@PE[pos,2*embed_dim:2*embed_dim+2])

    assert torch.allclose(R_k @ PE[pos, 2*embed_dim:2*embed_dim+2],PE[pos+k,2*embed_dim:2*embed_dim+2])

In [7]:
check(2,4,2)

tensor([0.0600, 0.9982])
tensor([0.0600, 0.9982])


## Solution 4 — SinusoidalPositionalEmbedding module

In [8]:
class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        pe = torch.zeros((max_len,d_model))

        omegas = 1 / torch.pow(10000,torch.arange(0,d_model,2)/d_model)
        angles = torch.arange(max_len)[:,None] * omegas[None,:]

        pe[:,::2] = torch.sin(angles)
        pe[:,1::2] = torch.cos(angles)

        self.register_buffer("pe",pe)

    def forward(self, token_embeddings):
        return token_embeddings+self.pe[:token_embeddings.shape[1],:] 

## Run the tests

In [9]:
from tests import run_sinusoidal_tests, run_sinusoidal_module_tests
run_sinusoidal_tests(sinusoidal_pe)
run_sinusoidal_module_tests(SinusoidalPositionalEmbedding)

Running sinusoidal_pe...
  ✓ Shape == (10, 64)
  ✓ PE[0, even cols] all 0 (sin of zero angle)
  ✓ PE[0, odd cols] all 1 (cos of zero angle)
  ✓ All values bounded in [-1, 1]
  ✓ Low dims oscillate faster than high dims
  ✓ Different positions have distinct codes
  ✓ Rotation property: R_k @ PE[pos] == PE[pos+k]

  All 7 checks passed ✓

Running SinusoidalPositionalEmbedding...
  ✓ Zero learnable parameters
  ✓ Buffer registered in state_dict
  ✓ Forward shape correct for seq_len < max_len
  ✓ Forward shape correct at full max_len
  ✓ forward(zeros) returns the PE values
  ✓ Buffer moves with module.to(device)

  All 6 checks passed ✓



True